# Gráfica de Energía del Proyectil 
### Physics for JavaScript Games, Animation & Simulations — Capítulo 5 (pp. 126–128)

Demuestra la **conservación de energía mecánica** en un proyectil lanzado verticalmente.

| Curva | Color | Fórmula | Comportamiento |
|-------|:---:|---------|----------------|
| Energía Potencial (PE) | 🔴 Rojo | $E_p = mgh$ | Sube mientras el proyectil asciende |
| Energía Cinética (KE) | 🔵 Azul | $E_k = \frac{1}{2}mv^2$ | Baja mientras el proyectil asciende |
| Energía Total | ⚫ Negro | $E_t = E_p + E_k$ | Constante — conservación de energía |

**Ecuaciones de movimiento** (lanzamiento vertical desde el suelo):
$$v = u - g \cdot t \qquad h = u \cdot t - \frac{1}{2}g \cdot t^2$$


In [2]:
# Celda 1 — Cargar p5.js
from IPython.display import HTML
HTML("""
<script src="https://cdnjs.cloudflare.com/ajax/libs/p5.js/1.9.0/p5.min.js"></script>
<p style="color:#4caf50;font-weight:bold;">✅ p5.js cargado</p>
""")

In [3]:
# Celda 2 — Gráfica de Energía del Proyectil (Figura 5-6 del libro)
from IPython.display import HTML

HTML("""
<div style="font-family:'Segoe UI',sans-serif; background:#0d1117; padding:20px;
            border-radius:12px; display:inline-block; border:1px solid #ddd;">

  <!-- Título -->
  <div style="font-size:14px; font-weight:bold; color:#333; margin-bottom:12px;">
    Figura 5-6 — Energía del Proyectil: PE (rojo) · KE (azul) · Total (negro)
  </div>

  <!-- Controles -->
  <div style="display:flex; gap:18px; flex-wrap:wrap; margin-bottom:12px;
              align-items:flex-end; font-size:12px; color:#555;">
    <div>
      <div style="margin-bottom:3px;">Vel. inicial u (px/s)</div>
      <input type="range" id="uSlider" min="20" max="100" step="5" value="50"
        style="width:110px;"
        oninput="document.getElementById('uVal').innerText=this.value">
      <span id="uVal">50</span>
    </div>
    <div>
      <div style="margin-bottom:3px;">Gravedad g</div>
      <input type="range" id="gSlider" min="2" max="20" step="1" value="10"
        style="width:110px;"
        oninput="document.getElementById('gVal').innerText=this.value">
      <span id="gVal">10</span>
    </div>
    <div>
      <div style="margin-bottom:3px;">Masa m</div>
      <input type="range" id="mSlider" min="0.5" max="3" step="0.5" value="1"
        style="width:110px;"
        oninput="document.getElementById('mVal').innerText=parseFloat(this.value).toFixed(1)">
      <span id="mVal">1.0</span>
    </div>
    <button onclick="energyReset()"
      style="background:#e94560;color:#fff;border:none;padding:7px 16px;
             border-radius:8px;cursor:pointer;font-size:12px;font-weight:bold;">▶ Reiniciar</button>
  </div>

  <!-- Canvas -->
  <div id="energy-canvas"></div>

  <!-- Leyenda -->
  <div style="display:flex; gap:22px; margin-top:8px; font-size:12px; color:#555;">
    <span><span style="color:#cc2200;font-weight:bold;">●</span> PE = mgh (Energía Potencial)</span>
    <span><span style="color:#0044cc;font-weight:bold;">●</span> KE = ½mv² (Energía Cinética)</span>
    <span><span style="color:#222;font-weight:bold;">—</span> Total = PE + KE (constante)</span>
  </div>

  <!-- Panel de datos -->
  <div id="energy-info"
    style="margin-top:8px; font-family:monospace; font-size:11px; color:#666;
           background:#f0f0f0; border-radius:6px; padding:6px 12px;"></div>
</div>

<script>
var energySketch = function(p) {

  // ── Dimensiones del canvas ──────────────────────────────
  var W = 620, H = 420;

  // ── Área de la gráfica (igual que el libro p.128) ───────
  // xorig, yorig = origen de la gráfica en píxeles del canvas
  var xOrig = 70, yOrig = 370;
  var gW    = 480, gH = 320;   // ancho y alto del área de la gráfica

  // ── Parámetros de la simulación (libro p.128) ───────────
  // m=1, g=10, u=50, t de 0 a 10 segundos, 101 puntos
  var N = 101;                 // número de puntos
  var tA = [], hA = [];
  var peA = [], keA = [], teA = [];

  // ── Estado de la animación ──────────────────────────────
  var n = 0;                   // índice del frame actual
  var running = true;

  function cfg() {
    return {
      m: parseFloat(document.getElementById('mSlider').value),
      g: parseFloat(document.getElementById('gSlider').value),
      u: parseFloat(document.getElementById('uSlider').value)
    };
  }

  // ── setupArrays — libro p.127 ───────────────────────────
  // Pre-calcula PE, KE y total para los 101 instantes de tiempo
  function setupArrays(c) {
    var tMax = 2 * c.u / c.g;    // tiempo de vuelo total
    for (var i = 0; i < N; i++) {
      tA[i]  = i * tMax / (N - 1);        // tiempo
      var t  = tA[i];
      var v  = c.u - c.g * t;             // v = u - g·t
      hA[i]  = c.u*t - 0.5*c.g*t*t;      // h = u·t - ½·g·t²
      if (hA[i] < 0) hA[i] = 0;          // no puede ser negativo
      peA[i] = c.m * c.g * hA[i];        // PE = m·g·h
      keA[i] = 0.5 * c.m * v * v;        // KE = ½·m·v²
      teA[i] = peA[i] + keA[i];          // Total = PE + KE
    }
  }

  function reset() {
    n       = 0;
    running = true;
    setupArrays(cfg());
  }
  window.energyReset = reset;

  // ── Conversión de coordenadas físicas → píxeles ─────────
  // El eje t va de 0 a tMax → mapeado a [xOrig, xOrig+gW]
  // El eje E va de 0 a Emax → mapeado a [yOrig, yOrig-gH]
  function tx(t) {
    var tMax = tA[N - 1];
    return xOrig + (t / tMax) * gW;
  }
  function ty(E) {
    var Emax = teA[0];      // energía total = constante = KE inicial
    return yOrig - (E / Emax) * gH;
  }

  // ── Dibujar ejes y cuadrícula ───────────────────────────
  function drawAxes() {
    // Fondo del área de la gráfica
    p.fill(255);
    p.stroke(200);
    p.strokeWeight(1);
    p.rect(xOrig, yOrig - gH, gW, gH);

    var tMax = tA[N - 1];
    var Emax = teA[0];

    // Cuadrícula vertical (tiempo)
    p.stroke(220); p.strokeWeight(0.8);
    for (var i = 1; i <= 10; i++) {
      var xg = xOrig + (i / 10) * gW;
      p.line(xg, yOrig, xg, yOrig - gH);
    }
    // Cuadrícula horizontal (energía)
    for (var j = 1; j <= 6; j++) {
      var yg = yOrig - (j / 6) * gH;
      p.line(xOrig, yg, xOrig + gW, yg);
    }

    // Ejes principales
    p.stroke(80); p.strokeWeight(1.5);
    p.line(xOrig, yOrig, xOrig + gW, yOrig);   // eje X
    p.line(xOrig, yOrig, xOrig, yOrig - gH);   // eje Y

    // Etiquetas eje X (tiempo)
    p.fill(80); p.noStroke(); p.textSize(11); p.textAlign(p.CENTER);
    for (var i = 0; i <= 10; i++) {
      var xLbl = xOrig + (i / 10) * gW;
      var tLbl = (i / 10) * tMax;
      p.text(tLbl.toFixed(1), xLbl, yOrig + 16);
    }

    // Etiquetas eje Y (energía)
    p.textAlign(p.RIGHT);
    for (var j = 0; j <= 6; j++) {
      var yLbl = yOrig - (j / 6) * gH;
      var eLbl = (j / 6) * Emax;
      p.text(Math.round(eLbl), xOrig - 6, yLbl + 4);
    }

    // Títulos de ejes
    p.textAlign(p.CENTER); p.textSize(12); p.fill(60);
    p.text('t (s)', xOrig + gW / 2, yOrig + 34);
    p.push();
    p.translate(xOrig - 42, yOrig - gH / 2);
    p.rotate(-p.HALF_PI);
    p.text('PE,  KE,  Total', 0, 0);
    p.pop();
  }

  // ── Dibujar línea completa de energía total ─────────────
  // El total es constante → se dibuja completo desde el inicio
  function drawTotalLine() {
    p.stroke(40); p.strokeWeight(1.8);
    p.noFill();
    p.beginShape();
    for (var i = 0; i < N; i++) {
      p.vertex(tx(tA[i]), ty(teA[i]));
    }
    p.endShape();
  }

  // ── Dibujar pelota (sincronizada con el gráfico) ─────────
  // La pelota se mueve en la zona derecha del canvas
  // Su altura refleja hA[n]
  function drawBall() {
    var c   = cfg();
    var hMax = c.u * c.u / (2 * c.g);   // altura máxima = u²/2g
    var bX  = xOrig + gW + 50;          // posición fija horizontal
    var bY  = yOrig - (hA[n] / hMax) * (gH * 0.85); // escalar altura
    var r   = 12;

    // Línea de trayectoria
    p.stroke(180); p.strokeWeight(1);
    p.line(bX, yOrig, bX, yOrig - gH * 0.85);

    // Sombra
    p.noStroke(); p.fill(0, 0, 0, 40);
    p.ellipse(bX + 3, bY + 3, r*2, r*2);

    // Cuerpo
    p.fill(30, 30, 30); p.stroke(80); p.strokeWeight(1.5);
    p.ellipse(bX, bY, r*2, r*2);

    // Punto de referencia en el eje X de la gráfica
    p.fill(200,50,50); p.noStroke();
    p.ellipse(tx(tA[n]), yOrig, 6, 6);

    // Línea vertical de tiempo actual
    p.stroke(180, 50, 50, 120); p.strokeWeight(1);
    p.line(tx(tA[n]), yOrig, tx(tA[n]), yOrig - gH);
  }

  p.setup = function() {
    p.createCanvas(W, H).parent('energy-canvas');
    p.frameRate(10);    // 10 fps como el libro (p.129: setTimeout 1000/10)
    reset();
  };

  p.draw = function() {
    p.background(250);

    drawAxes();
    drawTotalLine();

    // ── Dibujar puntos acumulados hasta el frame n ──────
    // PE — curva roja que sube (parábola hacia arriba)
    for (var i = 0; i <= n; i++) {
      p.noStroke();
      p.fill(204, 34, 0, 220);
      p.ellipse(tx(tA[i]), ty(peA[i]), 5, 5);
    }

    // KE — curva azul que baja (parábola hacia abajo)
    for (var i = 0; i <= n; i++) {
      p.noStroke();
      p.fill(0, 68, 204, 220);
      p.ellipse(tx(tA[i]), ty(keA[i]), 5, 5);
    }

    // Pelota animada
    drawBall();

    // Panel de datos
    if (n < N) {
      document.getElementById('energy-info').innerHTML =
        't = ' + tA[n].toFixed(2) + ' s' +
        ' &nbsp;|&nbsp; h = ' + hA[n].toFixed(1) + ' px' +
        ' &nbsp;|&nbsp; <span style="color:#cc2200;">PE = ' + peA[n].toFixed(1) + '</span>' +
        ' &nbsp;|&nbsp; <span style="color:#0044cc;">KE = ' + keA[n].toFixed(1) + '</span>' +
        ' &nbsp;|&nbsp; <b>Total = ' + teA[n].toFixed(1) + '</b>' +
        (Math.abs(n - Math.round(N/2)) < 2 ? ' &nbsp; ⬆️ punto más alto: KE = 0' : '');
    }

    // Avanzar el índice (animación)
    if (running) {
      n++;
      if (n >= N) {
        n = N - 1;
        running = false;
      }
    }
  };
};

new p5(energySketch);
</script>
""")

---
## Física de la Figura 5-6

### Ecuaciones del proyectil (lanzamiento vertical)
```
v = u - g·t          velocidad en cualquier instante
h = u·t - ½·g·t²     altura en cualquier instante
```

### Energías calculadas (libro p. 128)
$$E_p = mgh \qquad E_k = \tfrac{1}{2}mv^2 \qquad E_{total} = E_p + E_k = \text{constante}$$

### Lo que demuestra la gráfica
| Instante | Altura | PE | KE | Observación |
|----------|--------|----|----|-------------|
| $t = 0$ | 0 | 0 | máximo | Lanzamiento desde el suelo |
| $t = u/g$ | máxima | máximo | 0 | Punto más alto — velocidad cero |
| $t = 2u/g$ | 0 | 0 | máximo | Regresa al suelo |

> La línea negra horizontal constante demuestra la **conservación de energía mecánica**:  
> toda la energía cinética se convierte en potencial y viceversa, sin pérdidas (sin drag).
